# StyleBot - Entrega 2: integración de resultados verificados

**Coordinación y revisión:** Krysto Alberto Adan Jauregui Zuñiga  
**Integración documental:** Jacqueline Soto  
**Semilla común:** 42  
**Estado:** integración actualizada con resultados disponibles al 15 de noviembre de 2026.

Este notebook reúne los resultados reproducibles de datos, aumento, arquitecturas CNN, transfer learning e inferencia. La selección del modelo se mantiene basada únicamente en validación.

## 1. Protocolo común de datos

- Entrenamiento: 48,000 imágenes.
- Validación: 12,000 imágenes.
- Prueba: 10,000 imágenes oficiales.
- Entrada para CNN propias: `(28, 28, 1)`.
- Normalización: `float32` entre 0 y 1.
- División estratificada y semilla 42.
- Aumento aplicado solamente al entrenamiento.
- Validación y prueba sin transformaciones aleatorias.

La preparación completa y sus comprobaciones se encuentran en `00_datos_cnn.ipynb`.

## 2. Experimento controlado de aumento

Se comparó la misma CNN base con idéntica inicialización, división, optimizador, lote y ocho épocas. Bajo esta configuración, el aumento no mejoró la validación.

In [1]:
resultados_aumento = [
    {'condición': 'Sin aumento', 'accuracy_validación': 0.9188, 'f1_macro_validación': 0.9183, 'tiempo_s': 32.9020, 'parámetros': 402442},
    {'condición': 'Con aumento', 'accuracy_validación': 0.8790, 'f1_macro_validación': 0.8793, 'tiempo_s': 58.2762, 'parámetros': 402442},
]
print('Condición | Accuracy val. | F1 macro val. | Tiempo (s) | Parámetros')
for fila in resultados_aumento:
    print(f"{fila['condición']} | {fila['accuracy_validación']:.4f} | {fila['f1_macro_validación']:.4f} | {fila['tiempo_s']:.4f} | {fila['parámetros']:,}")
assert len({fila['parámetros'] for fila in resultados_aumento}) == 1
print('Experimento controlado verificado: misma arquitectura y parámetros.')

Condición | Accuracy val. | F1 macro val. | Tiempo (s) | Parámetros
Sin aumento | 0.9188 | 0.9183 | 32.9020 | 402,442
Con aumento | 0.8790 | 0.8793 | 58.2762 | 402,442
Experimento controlado verificado: misma arquitectura y parámetros.


## 3. Arquitecturas CNN propias y selección con validación

Las arquitecturas A, B y C fueron entrenadas con la misma división, semilla, lote, máximo de épocas y callbacks. `EarlyStopping` restauró los mejores pesos y `ReduceLROnPlateau` ajustó la tasa de aprendizaje cuando la pérdida de validación dejó de mejorar.

In [2]:
tabla_validacion = [
    {'modelo': 'Arquitectura A sin aumento', 'parámetros': 402442, 'tiempo_s': 49.07, 'val_loss': 0.2363, 'val_accuracy': 0.9213, 'val_f1_macro': 0.9215},
    {'modelo': 'Arquitectura A con aumento', 'parámetros': 402442, 'tiempo_s': 199.93, 'val_loss': 0.2925, 'val_accuracy': 0.9010, 'val_f1_macro': 0.9006},
    {'modelo': 'Arquitectura B', 'parámetros': 421930, 'tiempo_s': 251.71, 'val_loss': 0.2629, 'val_accuracy': 0.9025, 'val_f1_macro': 0.9013},
    {'modelo': 'Arquitectura C', 'parámetros': 1749034, 'tiempo_s': 360.68, 'val_loss': 0.1903, 'val_accuracy': 0.9321, 'val_f1_macro': 0.9319},
    {'modelo': 'MobileNetV2', 'parámetros': 2428362, 'tiempo_s': 316.98, 'val_loss': 1.5850, 'val_accuracy': 0.4172, 'val_f1_macro': 0.4261},
]
print('Modelo | Parámetros | Tiempo (s) | Val loss | Val accuracy | Val F1 macro')
for fila in tabla_validacion:
    print(f"{fila['modelo']} | {fila['parámetros']:,} | {fila['tiempo_s']:.2f} | {fila['val_loss']:.4f} | {fila['val_accuracy']:.4f} | {fila['val_f1_macro']:.4f}")
candidatas = [fila for fila in tabla_validacion if fila['modelo'].startswith('Arquitectura')]
mejor_cnn = max(candidatas, key=lambda fila: fila['val_f1_macro'])
assert mejor_cnn['modelo'] == 'Arquitectura C'
print(f"Mejor CNN propia por validación: {mejor_cnn['modelo']} - F1 macro {mejor_cnn['val_f1_macro']:.4f}")

Modelo | Parámetros | Tiempo (s) | Val loss | Val accuracy | Val F1 macro
Arquitectura A sin aumento | 402,442 | 49.07 | 0.2363 | 0.9213 | 0.9215
Arquitectura A con aumento | 402,442 | 199.93 | 0.2925 | 0.9010 | 0.9006
Arquitectura B | 421,930 | 251.71 | 0.2629 | 0.9025 | 0.9013
Arquitectura C | 1,749,034 | 360.68 | 0.1903 | 0.9321 | 0.9319
MobileNetV2 | 2,428,362 | 316.98 | 1.5850 | 0.4172 | 0.4261
Mejor CNN propia por validación: Arquitectura C - F1 macro 0.9319


### Selección

La Arquitectura C alcanzó el mejor F1 macro de validación (0.9319) y el mejor accuracy (0.9321). Por ello se seleccionó como CNN propia y se publicó como `modelos/stylebot_cnn_final.keras`. Esta decisión se tomó antes de considerar resultados del conjunto de prueba.

## 4. Guardado, carga e inferencia

Los modelos A, B y C se guardaron en formato `.keras`. El notebook `03_inferencia_integracion.ipynb` se ejecutó completamente y comprobó la carga, las predicciones, los tamaños, los tiempos y una sesión limpia.

In [3]:
tabla_inferencia = [
    {'modelo': 'Arquitectura A', 'tamaño_mb': 4.63, 'carga_s': 0.124, 'ms_por_imagen': 8.923, 'val_accuracy_recargada': 0.9213, 'val_f1_recargado': 0.9215},
    {'modelo': 'Arquitectura B', 'tamaño_mb': 4.88, 'carga_s': 0.263, 'ms_por_imagen': 16.827, 'val_accuracy_recargada': 0.8990, 'val_f1_recargado': 0.8984},
    {'modelo': 'Arquitectura C', 'tamaño_mb': 20.11, 'carga_s': 0.743, 'ms_por_imagen': 34.069, 'val_accuracy_recargada': 0.9321, 'val_f1_recargado': 0.9319},
]
print('Modelo | Tamaño MB | Carga s | ms/imagen | Accuracy recargada | F1 recargado')
for fila in tabla_inferencia:
    print(f"{fila['modelo']} | {fila['tamaño_mb']:.2f} | {fila['carga_s']:.3f} | {fila['ms_por_imagen']:.3f} | {fila['val_accuracy_recargada']:.4f} | {fila['val_f1_recargado']:.4f}")
print('Modelo más ligero:', min(tabla_inferencia, key=lambda fila: fila['tamaño_mb'])['modelo'])
print('Modelo con mayor F1 recargado:', max(tabla_inferencia, key=lambda fila: fila['val_f1_recargado'])['modelo'])

Modelo | Tamaño MB | Carga s | ms/imagen | Accuracy recargada | F1 recargado
Arquitectura A | 4.63 | 0.124 | 8.923 | 0.9213 | 0.9215
Arquitectura B | 4.88 | 0.263 | 16.827 | 0.8990 | 0.8984
Arquitectura C | 20.11 | 0.743 | 34.069 | 0.9321 | 0.9319
Modelo más ligero: Arquitectura A
Modelo con mayor F1 recargado: Arquitectura C


### Observación sobre B

La métrica recalculada al cargar `arquitectura_b.keras` (accuracy 0.8990 y F1 0.8984) difiere ligeramente de la tabla de entrenamiento (0.9025 y 0.9013). La diferencia no cambia la selección de C, pero debe confirmarse que B conserva los mejores pesos restaurados por `EarlyStopping`.

## 5. Transfer learning con MobileNetV2

La misma división y semilla se conservaron. Las imágenes se convirtieron de escala de grises a RGB, se redimensionaron de 28 por 28 a 32 por 32 y se escalaron al intervalo esperado por MobileNetV2. La base preentrenada se mantuvo congelada durante el entrenamiento de la cabeza clasificadora.

MobileNetV2 obtuvo 0.4172 de accuracy y 0.4261 de F1 macro en validación, muy por debajo de la Arquitectura C. El notebook de inferencia reconstruye la arquitectura sin pesos entrenados cuando el archivo MobileNetV2 no está disponible; esos tiempos y tamaño son referenciales y no sustituyen las métricas del entrenamiento real.

## 6. Comparación con el modelo clásico

El mejor modelo clásico de la Entrega 1 fue Regresión Logística con HOG y momentos de Hu, con accuracy 0.8841 y F1 macro 0.8839 en prueba. Esos valores se conservan como referencia histórica. La comparación final homogénea con C y MobileNetV2 se documentará en la evaluación final, sin ajustar modelos después de consultar prueba.

## 7. Control del conjunto de prueba

Se detectó que `02_metricas_comparacion.ipynb` ejecutó prueba antes de la fecha acordada. Esos resultados anticipados no se utilizan para seleccionar modelos ni cambiar hiperparámetros. La Arquitectura C ya había sido elegida mediante validación.

El lunes 16 se congelarán las configuraciones antes de registrar la evaluación final. Después de observar prueba no se permitirá modificar arquitecturas, aumento, callbacks ni parámetros.

## 8. Estado de contribuciones

| Responsable | Aportación verificada | Estado |
|---|---|---|
| Krysto | Datos, aumento, reproducibilidad e integración | Actualizado |
| Juan José | A/B/C, MobileNetV2, historiales y modelos | Completo |
| Ángel Uriel | Métricas y matrices | Requiere separar la prueba anticipada |
| Luis Ángel | Guardado, carga, inferencia y sesión limpia | Completo |
| Jacqueline | Informe e integración documental | Requiere agregar curvas y actualizar estados |

In [4]:
controles = [
    {'control': 'Semilla común 42', 'estado': 'Aprobado'},
    {'control': 'Aumento solo en entrenamiento', 'estado': 'Aprobado'},
    {'control': 'Selección de C con validación', 'estado': 'Aprobado'},
    {'control': 'Modelos A/B/C publicados', 'estado': 'Aprobado'},
    {'control': 'Inferencia en sesión limpia', 'estado': 'Aprobado'},
    {'control': 'Prueba reservada hasta el cierre', 'estado': 'Incidencia documentada'},
    {'control': 'Curvas dentro del informe', 'estado': 'Pendiente de documentación'},
]
for fila in controles:
    print(f"{fila['control']}: {fila['estado']}")
assert sum(fila['estado'] == 'Aprobado' for fila in controles) == 5
print('Revisión del líder registrada. La evaluación final no autoriza ajustes posteriores.')

Semilla común 42: Aprobado
Aumento solo en entrenamiento: Aprobado
Selección de C con validación: Aprobado
Modelos A/B/C publicados: Aprobado
Inferencia en sesión limpia: Aprobado
Prueba reservada hasta el cierre: Incidencia documentada
Curvas dentro del informe: Pendiente de documentación
Revisión del líder registrada. La evaluación final no autoriza ajustes posteriores.


## 9. Pendientes antes de la entrega

- Confirmar los mejores pesos del archivo B.
- Separar en el notebook de métricas las salidas de validación y prueba.
- Agregar curvas y matrices al informe.
- Registrar la evaluación final sin ajustes posteriores.
- Ejecutar este notebook completo desde cero el martes 17.
- Revisar conclusiones, contribuciones, referencias, README y requirements.

## Fuentes principales

- TensorFlow. Fashion-MNIST. https://www.tensorflow.org/api_docs/python/tf/keras/datasets/fashion_mnist/load_data
- TensorFlow. Data augmentation. https://www.tensorflow.org/tutorials/images/data_augmentation
- TensorFlow. Transfer learning and fine-tuning. https://www.tensorflow.org/tutorials/images/transfer_learning
- Keras. EarlyStopping. https://keras.io/api/callbacks/early_stopping/
- Sandler, M., Howard, A., Zhu, M., Zhmoginov, A. y Chen, L.-C. (2018). MobileNetV2: Inverted Residuals and Linear Bottlenecks. https://doi.org/10.1109/CVPR.2018.00474